In [1]:
import time
import functools


def log_execution(func):
    """Measure and log the execution time of a function."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_time = time.perf_counter() - start_time

        print(
            f"[LOG] Function '{func.__name__}' "
            f"executed in {elapsed_time:.6f} s"
        )

        return result

    return wrapper


@log_execution
def stream_sensor_data(
    readings: list,
    threshold: float,
    /,
    *,
    upper_limit: float = 60.0
):
    """Yield valid sensor readings within the specified range."""

    for reading in readings:
        # Ignore values that are not integers or floating-point numbers.
        if isinstance(reading, (int, float)):
            if threshold <= reading <= upper_limit:
                yield reading


# ============================================================
# MAIN TELEMETRY PIPELINE
# ============================================================

print("--- Starting Telemetry Pipeline ---")

raw_readings = [
    21.5,
    "ERR",
    28.0,
    14.2,
    None,
    31.0,
    "SENSOR FAIL",
    18.5,
    25.0
]

cutoff_threshold = 20.0

valid_readings = stream_sensor_data(
    raw_readings,
    cutoff_threshold
)

fahrenheit_readings = (
    round(celsius * 1.8 + 32, 2)
    for celsius in valid_readings
)

for fahrenheit in fahrenheit_readings:
    print(f"Processed Reading: {fahrenheit} F")


# ============================================================
# FR5: METADATA VERIFICATION
# ============================================================

assert stream_sensor_data.__name__ == "stream_sensor_data"
assert stream_sensor_data.__doc__

print("\n[+] Metadata introspection test passed successfully.")


# ============================================================
# FR5: POSITIONAL-ONLY PARAMETER TEST
# ============================================================

try:
    stream_sensor_data(
        readings=raw_readings,
        threshold=cutoff_threshold
    )
except TypeError as error:
    print("\nPositional-only parameter test:")
    print(error)


# ============================================================
# EDGE CASE (a)
# WHAT IS THE DECORATOR REALLY TIMING?
# ============================================================

print("\n--- Edge Case (a): Generator Timing ---")

large_readings = [25.0] * 1_000_000

print("\nCreating large generator:")

large_stream = stream_sensor_data(
    large_readings,
    20.0
)

start_time = time.perf_counter()

for reading in large_stream:
    pass

consumption_time = time.perf_counter() - start_time

print(
    f"Actual generator consumption time: "
    f"{consumption_time:.6f} s"
)

print(
    "\nThe decorator measures generator creation, "
    "while the for loop measures generator consumption."
)


# ============================================================
# EDGE CASE (b)
# BOOLEANS AND SPECIAL FLOATS
# ============================================================

print("\n--- Edge Case (b): Booleans and Special Floats ---")

boolean_readings = [True, False]

print("\nBoolean test:")

boolean_stream = stream_sensor_data(
    boolean_readings,
    0.0
)

for reading in boolean_stream:
    print(f"Boolean value emitted as reading: {reading}")


print("\nNaN test:")

nan_readings = [float("nan")]

nan_stream = stream_sensor_data(
    nan_readings,
    0.0
)

nan_found = False

for reading in nan_stream:
    nan_found = True
    print(f"NaN emitted: {reading}")

if not nan_found:
    print("NaN was not emitted.")


print("\nInfinity test:")

infinity_readings = [float("inf")]

infinity_stream = stream_sensor_data(
    infinity_readings,
    0.0
)

for reading in infinity_stream:
    print(f"Infinity emitted: {reading}")


# ============================================================
# EDGE CASE (c)
# ONE-SHOT STREAMS AND EMPTY STREAMS
# ============================================================

print("\n--- Edge Case (c): One-Shot Streams ---")

one_shot_readings = [25.0, 30.0, 35.0]

one_shot_stream = stream_sensor_data(
    one_shot_readings,
    20.0
)

print("\nFirst iteration:")

for reading in one_shot_stream:
    print(f"Reading: {reading}")


print("\nSecond iteration:")

for reading in one_shot_stream:
    print(f"Reading: {reading}")


print("\nTesting next() on exhausted generator:")

try:
    next(one_shot_stream)
except StopIteration:
    print("StopIteration raised because the generator is exhausted.")


# ============================================================
# EDGE CASE (c): EMPTY INPUT
# ============================================================

print("\n--- Empty Stream Test ---")

empty_readings = []

empty_stream = stream_sensor_data(
    empty_readings,
    20.0
)

empty_found = False

for reading in empty_stream:
    empty_found = True
    print(f"Reading: {reading}")

if not empty_found:
    print("Empty stream produced no readings.")


# ============================================================
# EDGE CASE (c): ALL INVALID INPUT
# ============================================================

print("\n--- All Invalid Input Test ---")

invalid_readings = [
    "ERR",
    None,
    "SENSOR FAIL",
    "INVALID"
]

invalid_stream = stream_sensor_data(
    invalid_readings,
    20.0
)

invalid_found = False

for reading in invalid_stream:
    invalid_found = True
    print(f"Reading: {reading}")

if not invalid_found:
    print("All-invalid stream produced no readings.")


# ============================================================
# EDGE CASE (d)
# METADATA WITHOUT functools.wraps
# ============================================================

print("\n--- Edge Case (d): Metadata Without functools.wraps ---")


def log_execution_without_wraps(func):
    """Decorator used to demonstrate metadata loss."""

    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_time = time.perf_counter() - start_time

        print(
            f"[LOG] Function '{func.__name__}' "
            f"executed in {elapsed_time:.6f} s"
        )

        return result

    return wrapper


@log_execution_without_wraps
def sample_function():
    """This is the original sample function."""
    return "Sample result"


print("\nFunction name without functools.wraps:")
print(sample_function.__name__)

print("\nFunction documentation without functools.wraps:")
print(sample_function.__doc__)


# ============================================================
# EDGE CASE (d)
# POSITIONAL-ONLY DESIGN EXPLANATION
# ============================================================

print("\nWhy use positional-only parameters?")

print(
    "The first two parameters can be kept positional-only so that "
    "their parameter names are not treated as part of the public API."
)

print(
    "This allows a library author to change internal parameter names "
    "later without breaking keyword-based calls."
)


# ============================================================
# BONUS TASK (+10)
# UPPER PLAUSIBILITY LIMIT
# ============================================================

print("\n--- Bonus Task ---")

bonus_readings = [
    21.5,
    999.9,
    28.0
]


# ------------------------------------------------------------
# Bonus (i): 999.9 rejected with default upper_limit = 60.0
# ------------------------------------------------------------

print("\nBonus Test 1: Default upper limit")

default_limit_stream = stream_sensor_data(
    bonus_readings,
    20.0
)

for reading in default_limit_stream:
    print(f"Accepted reading: {reading}")

print(
    "999.9 is rejected because the default upper limit is 60.0."
)


# ------------------------------------------------------------
# Bonus (ii): 999.9 accepted with upper_limit=1000.0
# ------------------------------------------------------------

print("\nBonus Test 2: upper_limit=1000.0")

expanded_limit_stream = stream_sensor_data(
    bonus_readings,
    20.0,
    upper_limit=1000.0
)

for reading in expanded_limit_stream:
    print(f"Accepted reading: {reading}")


# ------------------------------------------------------------
# Bonus (iii): Third positional argument must fail
# ------------------------------------------------------------

print("\nBonus Test 3: Third positional argument")

try:
    stream_sensor_data(
        bonus_readings,
        20.0,
        1000.0
    )
except TypeError as error:
    print(error)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n--- Assignment Tests Completed ---")

print("FR1: Notebook environment requirement satisfied.")
print("FR2: log_execution decorator implemented.")
print("FR3: Lazy sensor generator implemented.")
print("FR4: Generator expression conversion implemented.")
print("FR5: Metadata and positional-only tests implemented.")
print("Edge cases (a), (b), (c), and (d) tested.")
print("Bonus task implemented.")

--- Starting Telemetry Pipeline ---
[LOG] Function 'stream_sensor_data' executed in 0.000002 s
Processed Reading: 70.7 F
Processed Reading: 82.4 F
Processed Reading: 87.8 F
Processed Reading: 77.0 F

[+] Metadata introspection test passed successfully.

Positional-only parameter test:
stream_sensor_data() got some positional-only arguments passed as keyword arguments: 'readings, threshold'

--- Edge Case (a): Generator Timing ---

Creating large generator:
[LOG] Function 'stream_sensor_data' executed in 0.000003 s
Actual generator consumption time: 0.348153 s

The decorator measures generator creation, while the for loop measures generator consumption.

--- Edge Case (b): Booleans and Special Floats ---

Boolean test:
[LOG] Function 'stream_sensor_data' executed in 0.000001 s
Boolean value emitted as reading: True
Boolean value emitted as reading: False

NaN test:
[LOG] Function 'stream_sensor_data' executed in 0.000001 s
NaN was not emitted.

Infinity test:
[LOG] Function 'stream_sens